# NiyamTrace-X — PART 2 FIXED V2: Llama + Final Merge

This is the corrected second notebook for the split three-family external
validation run.

### What was fixed

1. **Part-1 restore bug:** Part 2 now accepts either the rolling
   `NTX_FINAL_RECOVERY_LATEST.zip` **or** the complete
   `NTX_FINAL_3FAMILY_PART1_GRANITE_QWEN_RESULTS.zip`.
2. **Reproducibility:** vLLM is pinned to 0.29.0, EvalScope/BFCL to 1.11.1,
   AgentDojo to commit `089ed468...`, and tau2-bench to commit `2174a603...`.
3. **Llama T4 startup:** the preferred context remains 8192 tokens; an explicit
   GPU/KV-capacity startup failure triggers one transparent 6144-token retry,
   which is recorded and does **not** silently strengthen the paper claim.
4. **Cached AgentDojo correctness:** a non-empty old CSV is no longer
   automatically treated as supported unless its saved slice status was
   actually supported.
5. **Fail-fast checks:** current CLI/model compatibility is verified before
   expensive benchmark execution.

### Model run in this notebook

`solidrust/Hermes-3-Llama-3.1-8B-AWQ`

The model is a 4-bit AWQ quantization of Hermes-3/Llama-3.1-8B and is served
with vLLM's `hermes` tool-call parser.

Run this notebook **after Part 1**. Keep the Part-1 results in
`MyDrive/NTX_FINAL_3FAMILY_CLOSURE/`, or upload the Part-1 result ZIP to
`/content` before rerunning Cell 1.


In [ ]:
# CELL 1 — FINAL T4 CONFIGURATION + ROBUST PART-1 RECOVERY (FIXED V2)

from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, time, random, hashlib, zipfile, shutil, subprocess, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED)
np.random.seed(SEED)

MODE=os.getenv("NTX_FINAL_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE","FULL"}

BASE=Path("/content/NTX_FINAL_3FAMILY_CLOSURE")
WORK=BASE/"work"
MODELS=BASE/"models"
RAW=BASE/"raw"
RESULTS=BASE/"results"
LOGS=BASE/"logs"
PAPER=BASE/"paper_integration"
ARCH=BASE/"archives"
TEMPLATES=BASE/"templates"

for p in [BASE,WORK,MODELS,RAW,RESULTS,LOGS,PAPER,ARCH,TEMPLATES]:
    p.mkdir(parents=True,exist_ok=True)

CFG={
    "SMOKE":{
        "bfcl_limit":3,
        "dojo_suites":["banking"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline"],
        "tau_tasks":1,
        "tau_steps":14,
        "max_model_len":4096,
        "gpu_memory_utilization":0.78,
    },
    "CLOSURE":{
        "bfcl_limit":25,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0","user_task_1"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":2,
        "tau_steps":30,
        "max_model_len":8192,
        "gpu_memory_utilization":0.84,
    },
    "FULL":{
        "bfcl_limit":75,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0","user_task_1","user_task_5"],
        "dojo_injection_tasks":["injection_task_0","injection_task_1"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":5,
        "tau_steps":40,
        "max_model_len":8192,
        "gpu_memory_utilization":0.86,
    },
}[MODE]

PORT=8000
LOCAL_BASE=f"http://127.0.0.1:{PORT}/v1"
LOCAL_KEY="EMPTY"
KEEP_MODEL_WEIGHTS=True
HF_TOKEN=os.getenv("HF_TOKEN","").strip()
CHECKPOINT=RESULTS/"FINAL_CHECKPOINT.json"

# -------------------------------------------------------------------
# Persistent Google Drive recovery.
#
# V1 only checked NTX_FINAL_RECOVERY_LATEST.zip. That was fragile because
# Part 1 ALSO writes NTX_FINAL_3FAMILY_PART1_GRANITE_QWEN_RESULTS.zip.
# V2 accepts either archive and validates the restored checkpoint.
# -------------------------------------------------------------------

USE_DRIVE=True
DRIVE_DIR=None
LATEST_DRIVE_ZIP=None
PART1_ZIP_NAME="NTX_FINAL_3FAMILY_PART1_GRANITE_QWEN_RESULTS.zip"

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_DIR=Path("/content/drive/MyDrive/NTX_FINAL_3FAMILY_CLOSURE")
        DRIVE_DIR.mkdir(parents=True,exist_ok=True)
        LATEST_DRIVE_ZIP=DRIVE_DIR/"NTX_FINAL_RECOVERY_LATEST.zip"
        print("Drive recovery:",DRIVE_DIR)
    except Exception as e:
        print("Drive recovery unavailable:",repr(e))
        DRIVE_DIR=None
        LATEST_DRIVE_ZIP=None

def _safe_extract_zip(zip_path,destination):
    """Extract only paths that remain inside destination."""
    zip_path=Path(zip_path)
    destination=Path(destination)
    destination.mkdir(parents=True,exist_ok=True)
    dest_resolved=destination.resolve()

    with zipfile.ZipFile(zip_path) as z:
        bad=z.testzip()
        if bad is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad}")

        for info in z.infolist():
            name=info.filename.replace("\\","/")
            if not name or name.endswith("/"):
                continue
            target=(destination/name).resolve()
            if not str(target).startswith(str(dest_resolved)+os.sep):
                raise RuntimeError(f"Unsafe ZIP path rejected: {name}")

        z.extractall(destination)

def _checkpoint_has_part1():
    try:
        state=json.loads(CHECKPOINT.read_text())
    except Exception:
        return False,{}

    required=["granite33_2b","qwen25_3b"]
    return all(x in state for x in required),state

def import_part1_archive(zip_path):
    """Import a Part-1/recovery ZIP and verify Granite + Qwen checkpoint keys."""
    zip_path=Path(zip_path)
    if not zip_path.exists():
        return False

    print("Restoring Part-1 evidence from:",zip_path)
    _safe_extract_zip(zip_path,BASE)

    ok,state=_checkpoint_has_part1()
    if not ok:
        raise RuntimeError(
            f"{zip_path.name} extracted, but results/FINAL_CHECKPOINT.json "
            "does not contain both granite33_2b and qwen25_3b."
        )

    print("✅ Part-1 checkpoint restored:",sorted(state.keys()))
    return True

def sync_recovery(label="latest"):
    """Persist small research outputs only; model weights are excluded."""
    local=ARCH/f"NTX_FINAL_RECOVERY_{label}.zip"
    if local.exists():
        local.unlink()

    with zipfile.ZipFile(local,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for folder_name,folder in [
            ("results",RESULTS),
            ("raw",RAW),
            ("logs",LOGS),
            ("paper_integration",PAPER),
        ]:
            if not folder.exists():
                continue
            for p in folder.rglob("*"):
                if p.is_file():
                    z.write(p,arcname=f"{folder_name}/{p.relative_to(folder)}")

    if LATEST_DRIVE_ZIP is not None:
        shutil.copy2(local,LATEST_DRIVE_ZIP)
        if label!="latest":
            stamp=DRIVE_DIR/f"NTX_FINAL_RECOVERY_{label}.zip"
            shutil.copy2(local,stamp)
        print("✅ Recovery synced to Drive:",LATEST_DRIVE_ZIP)

    return local

def restore_part1_evidence():
    # 0) Already present in the active runtime.
    ok,state=_checkpoint_has_part1()
    if ok:
        print("✅ Part-1 checkpoint already present locally.")
        return "LOCAL_CHECKPOINT"

    candidates=[]

    # 1) Preferred rolling recovery archive.
    if LATEST_DRIVE_ZIP is not None:
        candidates.append(LATEST_DRIVE_ZIP)

    # 2) Complete Part-1 results ZIP written by Part 1 Cell 10.
    if DRIVE_DIR is not None:
        candidates.append(DRIVE_DIR/PART1_ZIP_NAME)

    # 3) Manual/session upload fallback.
    candidates.extend([
        Path("/content")/PART1_ZIP_NAME,
        Path("/content/NTX_FINAL_RECOVERY_LATEST.zip"),
    ])

    for candidate in candidates:
        if candidate.exists():
            try:
                if import_part1_archive(candidate):
                    # Normalize into rolling recovery after successful fallback.
                    try:
                        sync_recovery("latest")
                    except Exception as e:
                        print("Recovery normalization warning:",repr(e))
                    return str(candidate)
            except Exception as e:
                print("Restore candidate rejected:",candidate,repr(e))

    return None

RESTORE_SOURCE=restore_part1_evidence()

print("MODE:",MODE)
print(json.dumps(CFG,indent=2))

if RESTORE_SOURCE is None:
    print(
        "\n⚠️ PART-1 EVIDENCE NOT FOUND YET.\n"
        "Expected one of:\n"
        "  MyDrive/NTX_FINAL_3FAMILY_CLOSURE/NTX_FINAL_RECOVERY_LATEST.zip\n"
        "  MyDrive/NTX_FINAL_3FAMILY_CLOSURE/"
        + PART1_ZIP_NAME + "\n"
        "You may also upload the Part-1 ZIP to /content and rerun this cell."
    )
else:
    print("\n✅ Part-1 restore source:",RESTORE_SOURCE)

In [ ]:
# CELL 2 — INSTALL LOCAL INFERENCE + UTILITIES (FIXED)
# Root-cause fix:
# vLLM is installed in a clean Python 3.12 uv environment instead of
# Colab's base Python environment. This prevents stale Colab TorchAudio/
# TorchVision packages from being imported against a different CUDA build.

def sh(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
    )

def save_log(name,p,cmd=None):
    text=""
    if cmd:
        text+="COMMAND\n"+" ".join(map(str,cmd))+"\n\n"
    text+="STDOUT\n"+(p.stdout or "")+"\n\nSTDERR\n"+(p.stderr or "")
    (LOGS/name).write_text(text,errors="ignore")

def ensure_base_package(import_name,pip_spec=None):
    try:
        __import__(import_name)
        return
    except Exception:
        pass
    spec=pip_spec or import_name
    p=sh([sys.executable,"-m","pip","install","-q","-U",spec])
    save_log(f"install_{import_name}.log",p)
    if p.returncode:
        print(p.stderr[-4000:])
        raise RuntimeError(f"Could not install {spec}")

# uv first.
if shutil.which("uv") is None:
    p=sh([sys.executable,"-m","pip","install","-q","-U","uv"])
    save_log("install_uv.log",p)
    if p.returncode:
        raise RuntimeError("uv installation failed")

# Lightweight clients remain in base runtime.
ensure_base_package("openai","openai")
ensure_base_package("huggingface_hub","huggingface_hub")
ensure_base_package("psutil","psutil")

from openai import OpenAI
from huggingface_hub import snapshot_download, HfApi

# ------------------------------------------------------------
# Isolated vLLM environment
# ------------------------------------------------------------

VLLM_ENV=WORK/"vllm_py312"
VLLM_PY=VLLM_ENV/"bin"/"python"

# Managed Python 3.12 is the stable isolated runtime.
p=sh(["uv","python","install","3.12"])
save_log("vllm_python312_install.log",p)

if not VLLM_ENV.exists():
    p=sh([
        "uv","venv",VLLM_ENV,
        "--python","3.12",
        "--seed",
        "--managed-python",
    ])
    save_log("vllm_venv_create.log",p)
    if p.returncode:
        raise RuntimeError("Could not create isolated vLLM Python 3.12 environment.")

# Install vLLM with uv selecting a CUDA/PyTorch backend compatible with
# the runtime driver. This does NOT mutate Colab's base torch/torchaudio.
p=sh([
    "uv","pip","install",
    "--python",VLLM_PY,
    "vllm==0.29.0",
    "--torch-backend=auto",
])
save_log("vllm_isolated_install.log",p)
if p.returncode:
    print(p.stderr[-6000:])
    raise RuntimeError("Isolated vLLM installation failed.")

# Critical verification: vLLM + torch import in the exact interpreter
# that will launch the API server. Also confirm a stale torchaudio package
# is not visible in this isolated environment.
verify_code = r"""
import importlib.util, json, torch, vllm
print(json.dumps({
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "vllm": vllm.__version__,
    "torchaudio_visible": importlib.util.find_spec("torchaudio") is not None,
    "torchvision_visible": importlib.util.find_spec("torchvision") is not None,
}))
"""

v=sh([VLLM_PY,"-c",verify_code])
save_log("vllm_isolated_verify.log",v,[VLLM_PY,"-c","<verification>"])
if v.returncode:
    print(v.stderr[-6000:])
    raise RuntimeError("vLLM isolated runtime verification failed.")

print("Isolated vLLM runtime:")
print(v.stdout.strip())
if '"vllm": "0.29.0"' not in v.stdout and '"vllm":"0.29.0"' not in v.stdout:
    raise RuntimeError(
        "Expected pinned vLLM 0.29.0 but verification reported a different version."
    )

# Record environment provenance.
fr=sh(["uv","pip","freeze","--python",VLLM_PY])
(LOGS/"vllm_isolated_freeze.txt").write_text(fr.stdout or "")

print("✅ Isolated vLLM environment is ready for the T4.")


In [ ]:
# CELL 2B — FAIL-FAST LOCAL vLLM RUNTIME DIAGNOSTIC
# This must pass BEFORE any model download begins.

diag = sh([
    VLLM_PY,
    "-c",
    (
        "import torch,vllm,importlib.util;"
        "print('TORCH',torch.__version__);"
        "print('CUDA',torch.version.cuda);"
        "print('VLLM',vllm.__version__);"
        "print('TORCHAUDIO_VISIBLE',importlib.util.find_spec('torchaudio') is not None)"
    ),
])

print(diag.stdout)
if diag.returncode:
    print(diag.stderr)
    raise RuntimeError("Isolated vLLM runtime diagnostic failed.")

if "TORCH " not in diag.stdout or "VLLM " not in diag.stdout:
    raise RuntimeError("Unexpected vLLM diagnostic output.")

print("✅ Runtime diagnostic passed. Model downloads may proceed.")


In [ ]:
# CELL 3 — VERIFY T4 + DEFINE THE THREE REQUIRED FAMILIES

def gpu_info():
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError(
            "No NVIDIA GPU attached. In Colab choose Runtime > Change runtime type > T4 GPU."
        )
    p=sh([
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ])
    if p.returncode:
        raise RuntimeError(p.stderr)
    line=p.stdout.strip().splitlines()[0]
    name,mem=line.rsplit(",",1)
    return name.strip(),float(mem.strip())/1024

GPU_NAME,VRAM_GB=gpu_info()
DISK_FREE_GB=shutil.disk_usage("/content").free/(1024**3)

print("GPU:",GPU_NAME)
print(f"VRAM: {VRAM_GB:.1f} GiB")
print(f"Disk free: {DISK_FREE_GB:.1f} GiB")

if "T4" not in GPU_NAME.upper():
    print("⚠️ Notebook is tuned for a 15 GB NVIDIA T4.")

MODEL_REGISTRY=[
    {
        "slug":"granite33_2b",
        "family":"Granite",
        "repo":"ibm-granite/granite-3.3-2b-instruct",
        "tool_parser":"granite",
        "min_vram_gb":7.0,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "extra_server_args":[],
    },
    {
        "slug":"qwen25_3b",
        "family":"Qwen",
        "repo":"Qwen/Qwen2.5-3B-Instruct",
        "tool_parser":"hermes",
        "min_vram_gb":7.5,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "extra_server_args":[],
    },
    {
        "slug":"hermes3_llama31_8b_awq",
        "family":"Llama",
        "repo":"solidrust/Hermes-3-Llama-3.1-8B-AWQ",
        "tool_parser":"hermes",
        "min_vram_gb":10.5,
        "full_bench":True,
        "counts_for_closure":True,
        "gated":False,
        "extra_server_args":["--quantization","awq"],
    },
]

for x in MODEL_REGISTRY:
    x["gpu_eligible"]=x["min_vram_gb"]<=VRAM_GB
    x["token_eligible"]=True

registry_df=pd.DataFrame(MODEL_REGISTRY)
display(registry_df[[
    "slug","family","repo","min_vram_gb",
    "gpu_eligible","full_bench","counts_for_closure"
]])
registry_df.to_csv(RESULTS/"00_final_model_registry.csv",index=False)

print("\nRequired closure families: Granite + Qwen + Llama")

In [ ]:
# CELL 4 — SET UP PINNED EXTERNAL BENCHMARKS

AGENTDOJO_COMMIT="089ed468cf3ed0322acc66b0211f26d9d90dbf60"
TAU2_COMMIT="2174a603f6d014ef94473ffa95957f6ce27100db"
EVALSCOPE_VERSION="1.11.1"

def clone_pinned(url,dest,commit):
    dest=Path(dest)

    if not dest.exists():
        p=sh(["git","clone","--filter=blob:none","--no-checkout",url,dest])
        save_log("clone_"+dest.name+".log",p)
        if p.returncode:
            raise RuntimeError(f"Clone failed: {url}")

    current=sh(["git","-C",dest,"rev-parse","HEAD"])
    current_sha=current.stdout.strip() if current.returncode==0 else ""

    if current_sha!=commit:
        p=sh(["git","-C",dest,"fetch","--depth","1","origin",commit])
        save_log("fetch_"+dest.name+".log",p)
        if p.returncode:
            raise RuntimeError(
                f"Could not fetch pinned commit {commit} for {url}"
            )

    # Always materialize the working tree. A --no-checkout clone can have
    # HEAD at the desired commit while the actual worktree is still empty.
    p=sh(["git","-C",dest,"checkout","--force","--detach",commit])
    save_log("checkout_"+dest.name+".log",p)
    if p.returncode:
        raise RuntimeError(
            f"Could not checkout pinned commit {commit} for {url}"
        )

    head=sh(["git","-C",dest,"rev-parse","HEAD"])
    if head.returncode or head.stdout.strip()!=commit:
        raise RuntimeError(
            f"Pinned checkout verification failed for {dest.name}: "
            f"expected {commit}, got {head.stdout.strip()}"
        )

    return commit

# ---------- BFCL / EvalScope isolated Python 3.11 ----------
BFENV=WORK/"bfcl_env"
sh(["uv","python","install","3.11"])

if not BFENV.exists():
    p=sh(["uv","venv",BFENV,"--python","3.11"])
    save_log("bfcl_venv.log",p)
    if p.returncode:
        raise RuntimeError("BFCL venv creation failed")

BFPY=BFENV/"bin"/"python"

p=sh([
    "uv","pip","install",
    "--python",BFPY,
    f"evalscope[bfcl]=={EVALSCOPE_VERSION}",
])
save_log("bfcl_install.log",p)
if p.returncode:
    print(p.stderr[-5000:])
    raise RuntimeError("BFCL/EvalScope install failed")

v=sh([
    BFPY,"-c",
    (
        "import evalscope;"
        "from evalscope import run_task;"
        "from evalscope.config import TaskConfig;"
        "print('BFCL_READY',getattr(evalscope,'__version__','unknown'))"
    )
])
save_log("bfcl_verify.log",v)

if v.returncode or "BFCL_READY" not in v.stdout:
    print(v.stderr[-5000:])
    raise RuntimeError("BFCL environment verification failed")

print(v.stdout.strip())

# ---------- AgentDojo: pin commit that contains openai-compatible ----------
DOJO=WORK/"agentdojo"
DOJO_COMMIT=clone_pinned(
    "https://github.com/ethz-spylab/agentdojo.git",
    DOJO,
    AGENTDOJO_COMMIT,
)

p=sh(["uv","sync"],cwd=DOJO)
save_log("dojo_sync.log",p)
if p.returncode:
    print(p.stderr[-5000:])
    raise RuntimeError("AgentDojo uv sync failed")

h=sh([
    "uv","run","python",
    "-m","agentdojo.scripts.benchmark",
    "--help",
],cwd=DOJO)

save_log("dojo_help.log",h)
ht=(h.stdout or "")+(h.stderr or "")

for token in [
    "openai-compatible",
    "--model-id",
    "--force-rerun",
    "--suite",
    "--user-task",
    "--injection-task",
    "--max-workers",
]:
    if token not in ht:
        raise RuntimeError(f"Pinned AgentDojo CLI missing {token}")

# ---------- tau2 / tau3: pin exact revision ----------
TAU=WORK/"tau2-bench"
TAU_COMMIT=clone_pinned(
    "https://github.com/sierra-research/tau2-bench.git",
    TAU,
    TAU2_COMMIT,
)

p=sh(["uv","sync"],cwd=TAU)
save_log("tau_sync.log",p)
if p.returncode:
    print(p.stderr[-5000:])
    raise RuntimeError("tau2 uv sync failed")

TAUPY=TAU/".venv"/"bin"/"python"

p=sh([
    "uv","pip","install",
    "--python",TAUPY,
    "websockets","soundfile",
],cwd=TAU)
save_log("tau_extra_deps.log",p)

v=sh([
    TAUPY,"-c",
    "import websockets,soundfile;print('TAU_READY')"
],cwd=TAU)

if v.returncode or "TAU_READY" not in v.stdout:
    print(v.stderr[-5000:])
    raise RuntimeError("tau2 dependency verification failed")

tau_help=sh(["uv","run","tau2","run","--help"],cwd=TAU)
save_log("tau_run_help.log",tau_help)
tau_ht=(tau_help.stdout or "")+(tau_help.stderr or "")

for token in [
    "--num-tasks",
    "--max-steps",
    "--auto-resume",
    "--agent-llm-args",
    "--user-llm-args",
    "--save-to",
    "--num-trials",
]:
    if token not in tau_ht:
        raise RuntimeError(f"Pinned tau2 CLI missing {token}")

BENCHMARK_VERSIONS={
    "evalscope_version":EVALSCOPE_VERSION,
    "agentdojo_commit":DOJO_COMMIT,
    "tau2_commit":TAU_COMMIT,
}

(RESULTS/"01_benchmark_versions.json").write_text(
    json.dumps(BENCHMARK_VERSIONS,indent=2)
)

print("✅ Pinned BFCL, AgentDojo, and tau2 environments ready.")
print(json.dumps(BENCHMARK_VERSIONS,indent=2))

In [ ]:
# CELL 5 — T4 vLLM SERVER V2 (DYNAMIC CLI COMPATIBILITY)

from huggingface_hub import snapshot_download, HfApi

SERVER=None

def load_checkpoint():
    try:
        return json.loads(CHECKPOINT.read_text())
    except Exception:
        return {}

def save_checkpoint(state):
    CHECKPOINT.write_text(json.dumps(state,indent=2,default=str))
    try:
        sync_recovery("latest")
    except Exception as e:
        print("Recovery sync warning:",repr(e))

STATE=load_checkpoint()

def remote_model_info(repo):
    api=HfApi(token=HF_TOKEN or None)
    info=api.model_info(repo,files_metadata=True)
    size=sum((getattr(s,"size",0) or 0) for s in info.siblings)/(1024**3)
    return info.sha,size

def download_model(spec):
    if spec.get("gated") and not HF_TOKEN:
        raise RuntimeError("GATED_MODEL_NO_HF_TOKEN")

    revision,size_gb=remote_model_info(spec["repo"])
    free=shutil.disk_usage("/content").free/(1024**3)

    required=max(3.0,size_gb*1.06+2.0)
    if free<required:
        raise RuntimeError(
            f"Not enough disk for {spec['repo']}: need ~{required:.1f} GiB, have {free:.1f}"
        )

    print(f"Downloading/reusing {spec['repo']} ({size_gb:.1f} GiB remote files)")

    path=snapshot_download(
        repo_id=spec["repo"],
        revision=revision,
        token=HF_TOKEN or None,
        cache_dir=str(MODELS/"hf_cache"),
    )

    return Path(path),revision,size_gb

def stop_server():
    global SERVER

    if SERVER is not None:
        try:
            SERVER.terminate()
            SERVER.wait(timeout=20)
        except Exception:
            try:
                SERVER.kill()
            except Exception:
                pass
        SERVER=None

    subprocess.run(
        ["pkill","-f","vllm.entrypoints.openai.api_server"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    time.sleep(4)
    gc.collect()

# ------------------------------------------------------------
# Detect the actual vLLM 0.29+ API-server CLI before serving.
# This prevents stale flags from killing every model.
# ------------------------------------------------------------

help_proc=sh([
    VLLM_PY,
    "-m","vllm.entrypoints.openai.api_server",
    "--help",
])

VLLM_SERVER_HELP=(help_proc.stdout or "")+(help_proc.stderr or "")

if help_proc.returncode not in (0,):
    print(VLLM_SERVER_HELP[-4000:])
    raise RuntimeError("Could not read vLLM API-server help.")

def cli_has(flag):
    return flag in VLLM_SERVER_HELP

for required in [
    "--model",
    "--served-model-name",
    "--gpu-memory-utilization",
    "--max-model-len",
    "--max-num-seqs",
    "--enforce-eager",
    "--enable-auto-tool-choice",
    "--tool-call-parser",
]:
    if not cli_has(required):
        raise RuntimeError(f"Installed vLLM CLI is missing required flag: {required}")

print("✅ vLLM CLI compatibility check passed.")

def _start_server_once(spec,model_path,max_model_len):
    global SERVER
    stop_server()

    logfile=LOGS/f"server_{spec['slug']}_{max_model_len}.log"
    fh=open(logfile,"w")

    cmd=[
        VLLM_PY,
        "-m","vllm.entrypoints.openai.api_server",
        "--model",str(model_path),
        "--served-model-name",spec["slug"],
        "--host","127.0.0.1",
        "--port",str(PORT),

        "--dtype","half",
        "--gpu-memory-utilization",str(CFG["gpu_memory_utilization"]),
        "--max-model-len",str(max_model_len),
        "--max-num-seqs","1",
        "--enforce-eager",

        "--enable-auto-tool-choice",
        "--tool-call-parser",spec["tool_parser"],
    ]

    if cli_has("--no-enable-log-requests"):
        cmd.append("--no-enable-log-requests")

    extras=list(spec.get("extra_server_args",[]))
    i=0
    while i<len(extras):
        token=extras[i]
        if token.startswith("--"):
            if not cli_has(token):
                print("Skipping unsupported optional vLLM flag:",token)
                if i+1<len(extras) and not str(extras[i+1]).startswith("--"):
                    i+=2
                else:
                    i+=1
                continue

            cmd.append(token)
            if i+1<len(extras) and not str(extras[i+1]).startswith("--"):
                cmd.append(str(extras[i+1]))
                i+=2
            else:
                i+=1
        else:
            cmd.append(str(token))
            i+=1

    print("Starting vLLM:", " ".join(map(str,cmd)))

    SERVER=subprocess.Popen(
        [str(x) for x in cmd],
        stdout=fh,
        stderr=subprocess.STDOUT,
        cwd=str(BASE),
        env=os.environ.copy(),
    )

    client=OpenAI(api_key=LOCAL_KEY,base_url=LOCAL_BASE)

    deadline=time.time()+420
    last_error=""

    while time.time()<deadline:
        if SERVER.poll() is not None:
            fh.flush()
            tail=logfile.read_text(errors="ignore")[-12000:]
            raise RuntimeError(
                f"VLLM_STARTUP_FAILED|max_model_len={max_model_len}\n"+tail
            )

        try:
            models=client.models.list()
            if models.data:
                print("Server ready:",models.data[0].id)
                return client
        except Exception as e:
            last_error=repr(e)

        time.sleep(5)

    stop_server()
    raise RuntimeError(
        f"VLLM_STARTUP_TIMEOUT|max_model_len={max_model_len}|{last_error}"
    )

def start_server(spec,model_path):
    requested=int(CFG["max_model_len"])

    # First try the scientifically preferred 8192-token context.
    try:
        client=_start_server_once(spec,model_path,requested)
        spec["_served_max_model_len"]=requested
        return client
    except RuntimeError as e:
        msg=str(e)
        stop_server()

        # Only use the fallback for explicit GPU/KV capacity failures.
        capacity_markers=[
            "CUDA out of memory",
            "out of memory",
            "KV cache",
            "maximum sequence length",
            "max seq len",
            "No available memory for the cache blocks",
        ]

        if spec["slug"]=="hermes3_llama31_8b_awq" and any(
            x.lower() in msg.lower() for x in capacity_markers
        ):
            fallback=6144
            print(
                f"⚠️ 8192-token Llama server did not fit on this T4. "
                f"Retrying once at {fallback} tokens."
            )
            client=_start_server_once(spec,model_path,fallback)
            spec["_served_max_model_len"]=fallback
            (RESULTS/"LLAMA_CONTEXT_FALLBACK.json").write_text(
                json.dumps({
                    "requested_max_model_len":requested,
                    "served_max_model_len":fallback,
                    "reason":"T4 vLLM startup capacity failure at requested context",
                    "scientific_note":(
                        "Any benchmark slice exceeding the served context must "
                        "remain incomplete; the fallback does not justify a full "
                        "closure claim by itself."
                    ),
                },indent=2)
            )
            return client

        raise

def tool_preflight(client,spec):
    result={
        "slug":spec["slug"],
        "family":spec["family"],
        "repo":spec["repo"],
        "chat_ok":False,
        "auto_tool_ok":False,
        "forced_tool_ok":False,
        "error":"",
    }

    tool=[{
        "type":"function",
        "function":{
            "name":"lookup_order",
            "description":"Look up an order by ID",
            "parameters":{
                "type":"object",
                "properties":{"order_id":{"type":"string"}},
                "required":["order_id"],
            },
        },
    }]

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Reply exactly OK."}],
            temperature=0,max_tokens=32,
        )
        result["chat_ok"]=bool(r.choices)
    except Exception as e:
        result["error"]="CHAT:"+repr(e)
        return result

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{
                "role":"user",
                "content":"Use lookup_order to look up order A123 before answering."
            }],
            tools=tool,
            tool_choice="auto",
            temperature=0,
            max_tokens=192,
        )

        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["auto_tool_ok"]=bool(
            calls and calls[0].function.name=="lookup_order"
        )

    except Exception as e:
        result["error"]+=" AUTO:"+repr(e)

    try:
        r=client.chat.completions.create(
            model=spec["slug"],
            messages=[{"role":"user","content":"Call lookup_order for A123."}],
            tools=tool,
            tool_choice={
                "type":"function",
                "function":{"name":"lookup_order"},
            },
            temperature=0,
            max_tokens=192,
        )

        calls=getattr(r.choices[0].message,"tool_calls",None) if r.choices else None
        result["forced_tool_ok"]=bool(
            calls and calls[0].function.name=="lookup_order"
        )

    except Exception as e:
        result["error"]+=" FORCED:"+repr(e)

    return result

print("✅ T4 vLLM server helper V2 ready.")

In [ ]:
# CELL 6 — RESUMABLE BENCHMARK RUNNERS

def safe_run(cmd,cwd=None,env=None,timeout=1800):
    try:
        return sh(cmd,cwd=cwd,env=env,timeout=timeout),None
    except subprocess.TimeoutExpired as e:
        class R:
            returncode=124
            stdout=(e.stdout or "") if isinstance(e.stdout,str) else ""
            stderr=((e.stderr or "") if isinstance(e.stderr,str) else "")+"\nTIMEOUT"
        return R(),"TIMEOUT"

def parse_bfcl(root,spec):
    rec=[]
    root=Path(root)
    for p in root.rglob("*"):
        if not p.is_file():continue
        rel=str(p.relative_to(root))
        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)
                for c in df.columns:
                    if any(k in str(c).lower() for k in ["accuracy","score"]):
                        for v in pd.to_numeric(df[c],errors="coerce").dropna():
                            rec.append({
                                "benchmark":"BFCL-v4","slug":spec["slug"],
                                "family":spec["family"],"model":spec["repo"],
                                "slice":rel,"metric":str(c),"score":float(v),
                            })
            elif p.suffix.lower() in {".json",".jsonl"}:
                texts=(p.read_text(errors="ignore").splitlines()
                       if p.suffix.lower()==".jsonl" else [p.read_text(errors="ignore")])
                for txt in texts:
                    try:o=json.loads(txt)
                    except Exception:continue
                    stack=[("",o)]
                    while stack:
                        path,x=stack.pop()
                        if isinstance(x,dict):
                            for k,v in x.items():
                                q=f"{path}.{k}" if path else str(k)
                                if isinstance(v,(dict,list)):
                                    stack.append((q,v))
                                elif isinstance(v,(int,float)) and any(
                                    t in str(k).lower() for t in ["accuracy","score"]
                                ):
                                    rec.append({
                                        "benchmark":"BFCL-v4","slug":spec["slug"],
                                        "family":spec["family"],"model":spec["repo"],
                                        "slice":rel,"metric":q,"score":float(v),
                                    })
                        elif isinstance(x,list):
                            for i,v in enumerate(x):stack.append((f"{path}[{i}]",v))
        except Exception:
            pass
    return pd.DataFrame(rec).drop_duplicates() if rec else pd.DataFrame()

def run_bfcl(spec):
    out=RAW/"bfcl"/spec["slug"]
    out.mkdir(parents=True,exist_ok=True)
    result_file=RESULTS/f"bfcl_{spec['slug']}.csv"

    if result_file.exists() and result_file.stat().st_size>2:
        old=pd.read_csv(result_file)
        if len(old):
            return "SUPPORTED_REUSED",old,0

    runner=WORK/f"run_bfcl_{spec['slug']}.py"
    runner.write_text(f"""from evalscope import run_task
from evalscope.config import TaskConfig
cfg=TaskConfig(
 model={spec['slug']!r},
 api_url={LOCAL_BASE!r},
 api_key={LOCAL_KEY!r},
 eval_type='openai_api',
 datasets=['bfcl_v4'],
 work_dir={str(out)!r},
 limit={CFG['bfcl_limit']!r},
 seed={SEED},
 generation_config={{'temperature':0.0,'max_tokens':768,'timeout':120}},
 dataset_args={{'bfcl_v4':{{'extra_params':{{'is_fc_model':True}}}}}}
)
run_task(task_cfg=cfg)
""")

    p,timeout=safe_run([BFPY,runner],cwd=out,timeout=1800)
    save_log(f"bfcl_{spec['slug']}.log",p,[BFPY,runner])
    d=parse_bfcl(out,spec)
    d.to_csv(result_file,index=False)
    st="SUPPORTED" if len(d) else ("TIMEOUT" if timeout else "FAILED")
    try: sync_recovery(f"{spec['slug']}_after_bfcl")
    except Exception as e: print("Recovery sync warning:",repr(e))
    return st,d,p.returncode

def parse_dojo(root,spec,suite):
    rec=[]
    for p in Path(root).rglob("*.json"):
        try:o=json.loads(p.read_text())
        except Exception:continue
        if not isinstance(o,dict):continue
        u=o.get("utility")
        s=o.get("security")
        if not isinstance(u,bool) and not isinstance(s,bool):continue
        rec.append({
            "benchmark":"AgentDojo","slug":spec["slug"],
            "family":spec["family"],"model":spec["repo"],"suite":suite,
            "utility":np.nan if not isinstance(u,bool) else int(u),
            "security":np.nan if not isinstance(s,bool) else int(s),
            "error":o.get("error"),"source_file":str(p),
        })
    return pd.DataFrame(rec)

def run_dojo(spec):
    model_file=RESULTS/f"dojo_{spec['slug']}.csv"
    slice_file=RESULTS/f"dojo_status_{spec['slug']}.csv"

    if model_file.exists() and slice_file.exists():
        old=pd.read_csv(model_file)
        old_status=pd.read_csv(slice_file)
        if len(old) and len(old_status) and old_status.status.astype(str).str.startswith("SUPPORTED").all():
            return "SUPPORTED_REUSED",old,old_status

    parts=[]
    statuses=[]
    env=os.environ.copy()
    env["OPENAI_COMPATIBLE_BASE_URL"]=LOCAL_BASE
    env["OPENAI_COMPATIBLE_API_KEY"]=LOCAL_KEY

    for suite in CFG["dojo_suites"]:
        out=RAW/"agentdojo"/spec["slug"]/suite
        out.mkdir(parents=True,exist_ok=True)
        per_slice=RESULTS/f"dojo_{spec['slug']}_{suite}.csv"

        if per_slice.exists() and slice_file.exists():
            try:
                cached=pd.read_csv(per_slice)
                previous_status=pd.read_csv(slice_file)
                prior=previous_status[previous_status["suite"]==suite]
                prior_ok=(
                    len(prior)>0
                    and prior["status"].astype(str).str.startswith("SUPPORTED").all()
                )
            except Exception:
                cached=pd.DataFrame()
                prior_ok=False

            if len(cached) and prior_ok:
                parts.append(cached)
                statuses.append({
                    "suite":suite,
                    "status":"SUPPORTED_REUSED",
                    "valid":int(
                        ((cached.utility.notna())|(cached.security.notna())).sum()
                    ),
                    "errors":int(cached.error.notna().sum()) if "error" in cached else 0,
                    "returncode":0,
                })
                continue

        cmd=[
            "uv","run","python","-m","agentdojo.scripts.benchmark",
            "--model","openai-compatible","--model-id",spec["slug"],
            "--suite",suite,"--attack","important_instructions",
            "--logdir",str(out),"--force-rerun","--max-workers","1",
        ]
        if CFG["dojo_user_tasks"] is not None:
            for x in CFG["dojo_user_tasks"]:cmd += ["--user-task",x]
        if CFG["dojo_injection_tasks"] is not None:
            for x in CFG["dojo_injection_tasks"]:cmd += ["--injection-task",x]

        p,timeout=safe_run(cmd,cwd=DOJO,env=env,timeout=1200)
        save_log(f"dojo_{spec['slug']}_{suite}.log",p,cmd)
        d=parse_dojo(out,spec,suite)
        if len(d):
            d.to_csv(per_slice,index=False)
            parts.append(d)

        valid=int(((d.utility.notna())|(d.security.notna())).sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0
        st=("SUPPORTED" if valid>0 and errors==0 else
            "PARTIAL" if valid>0 else
            "TIMEOUT" if timeout else "FAILED")
        statuses.append({
            "suite":suite,"status":st,"valid":valid,
            "errors":errors,"returncode":p.returncode,
        })
        try: sync_recovery(f"{spec['slug']}_dojo_{suite}")
        except Exception as e: print("Recovery sync warning:",repr(e))

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    ss=pd.DataFrame(statuses)
    cases.to_csv(model_file,index=False)
    ss.to_csv(slice_file,index=False)
    overall=("SUPPORTED" if len(ss) and ss.status.astype(str).str.startswith("SUPPORTED").all()
             else "PARTIAL" if len(cases) else "FAILED")
    return overall,cases,ss

def parse_tau(path,spec,domain):
    try:o=json.loads(Path(path).read_text())
    except Exception:return pd.DataFrame()

    if isinstance(o,list):sims=o
    elif isinstance(o,dict):
        sims=next((o[k] for k in ["simulations","results","trajectories"]
                   if isinstance(o.get(k),list)),[])
    else:sims=[]

    rec=[]
    for i,s in enumerate(sims):
        if not isinstance(s,dict):continue
        reward=None
        if isinstance(s.get("reward_info"),dict) and isinstance(
            s["reward_info"].get("reward"),(int,float,bool)
        ):
            reward=float(s["reward_info"]["reward"])
        elif isinstance(s.get("reward"),(int,float,bool)):
            reward=float(s["reward"])
        err=s.get("error")
        if err is None and isinstance(s.get("info"),dict):err=s["info"].get("error")
        rec.append({
            "benchmark":"tau3","slug":spec["slug"],"family":spec["family"],
            "model":spec["repo"],"domain":domain,"trajectory_index":i,
            "task_id":s.get("task_id"),"reward":reward,"error":err,
        })
    return pd.DataFrame(rec)

def run_tau(spec):
    model_file=RESULTS/f"tau_{spec['slug']}.csv"
    slice_file=RESULTS/f"tau_status_{spec['slug']}.csv"

    if model_file.exists() and slice_file.exists():
        old=pd.read_csv(model_file)
        old_status=pd.read_csv(slice_file)
        if len(old) and len(old_status) and old_status.status.astype(str).str.startswith("SUPPORTED").all():
            return "SUPPORTED_REUSED",old,old_status

    parts=[]
    statuses=[]
    env=os.environ.copy()
    env["OPENAI_API_KEY"]=LOCAL_KEY
    env["OPENAI_API_BASE"]=LOCAL_BASE

    for domain in CFG["tau_domains"]:
        per_slice=RESULTS/f"tau_{spec['slug']}_{domain}.csv"
        if per_slice.exists():
            cached=pd.read_csv(per_slice)
            if len(cached) and cached.reward.notna().any():
                parts.append(cached)
                statuses.append({
                    "domain":domain,"status":"SUPPORTED_REUSED",
                    "evaluated":int(cached.reward.notna().sum()),
                    "errors":int(cached.error.notna().sum()) if "error" in cached else 0,
                    "returncode":0,
                })
                continue

        run_name=f"ntx_t4_{spec['slug']}_{domain}"
        live=TAU/"data"/"simulations"/run_name
        archive=RAW/"tau"/spec["slug"]/domain

        llm_args=json.dumps({
            "api_base":LOCAL_BASE,
            "api_key":LOCAL_KEY,
            "temperature":0.0,
            "max_tokens":768,
        })

        cmd=[
            "uv","run","tau2","run",
            "--domain",domain,
            "--agent-llm","openai/"+spec["slug"],
            "--user-llm","openai/"+spec["slug"],
            "--agent-llm-args",llm_args,
            "--user-llm-args",llm_args,
            "--num-trials","1",
            "--task-split-name","base",
            "--max-steps",str(CFG["tau_steps"]),
            "--max-errors","3",
            "--max-concurrency","1",
            "--max-retries","1",
            "--retry-delay","2",
            "--seed",str(SEED),
            "--save-to",run_name,
            "--auto-resume","--verbose-logs","--llm-log-mode","latest",
        ]
        if CFG["tau_tasks"] is not None:cmd += ["--num-tasks",str(CFG["tau_tasks"])]

        p,timeout=safe_run(cmd,cwd=TAU,env=env,timeout=1500)
        save_log(f"tau_{spec['slug']}_{domain}.log",p,cmd)

        if live.exists():
            if archive.exists():shutil.rmtree(archive)
            shutil.copytree(live,archive)

        d=(parse_tau(archive/"results.json",spec,domain)
           if (archive/"results.json").exists() else pd.DataFrame())
        if len(d):
            d.to_csv(per_slice,index=False)
            parts.append(d)

        n=int(d.reward.notna().sum()) if len(d) else 0
        errors=int(d.error.notna().sum()) if len(d) else 0
        st=("SUPPORTED" if n>0 and errors==0 else
            "PARTIAL" if n>0 else
            "TIMEOUT" if timeout else "FAILED")
        statuses.append({
            "domain":domain,"status":st,"evaluated":n,
            "errors":errors,"returncode":p.returncode,
        })
        try: sync_recovery(f"{spec['slug']}_tau_{domain}")
        except Exception as e: print("Recovery sync warning:",repr(e))

    cases=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    ss=pd.DataFrame(statuses)
    cases.to_csv(model_file,index=False)
    ss.to_csv(slice_file,index=False)
    overall=("SUPPORTED" if len(ss) and ss.status.astype(str).str.startswith("SUPPORTED").all()
             else "PARTIAL" if len(cases) else "FAILED")
    return overall,cases,ss

print("✅ Resumable benchmark runners ready.")


In [ ]:
# CELL 7 — RUN-ONE-MODEL HELPER (T4 RESUMABLE)

def get_spec(slug):
    for s in MODEL_REGISTRY:
        if s["slug"]==slug:return s
    raise KeyError(slug)

def mark_model(slug,**updates):
    global STATE
    STATE.setdefault(slug,{})
    STATE[slug].update(updates)
    STATE[slug]["updated_utc"]=datetime.now(timezone.utc).isoformat()
    save_checkpoint(STATE)

def run_one_model(slug):
    spec=get_spec(slug)
    print("\n"+"="*96)
    print("T4 MODEL:",spec["repo"],"| family:",spec["family"])
    print("="*96)

    if not spec["gpu_eligible"]:
        mark_model(slug,status="SKIPPED_VRAM")
        print("SKIP: configured VRAM threshold exceeds this T4.")
        return
    if not spec["token_eligible"]:
        mark_model(slug,status="SKIPPED_GATED")
        print("SKIP: gated model. Set HF_TOKEN only if you already have access.")
        return

    prev=STATE.get(slug,{})
    if prev.get("status") in {"ERROR","SKIPPED_TOOL_PREFLIGHT","PARTIAL_COMPLETE"}:
        print("Resuming previously incomplete model:", prev.get("status"))
        # Per-benchmark/per-slice CSV caches are preserved and reused.
    if prev.get("status")=="COMPLETE":
        print("✅ Already complete according to checkpoint.")
        return

    local_path=None
    try:
        mark_model(slug,status="DOWNLOADING")
        local_path,revision,size_gb=download_model(spec)
        mark_model(slug,status="DOWNLOADED",revision=revision,remote_size_gb=round(size_gb,3))

        mark_model(slug,status="STARTING_SERVER")
        client=start_server(spec,local_path)
        mark_model(slug,status="SERVER_READY")

        pf=tool_preflight(client,spec)
        (RESULTS/f"preflight_{slug}.json").write_text(json.dumps(pf,indent=2))
        mark_model(
            slug,status="PREFLIGHT_DONE",
            chat_ok=pf["chat_ok"],auto_tool_ok=pf["auto_tool_ok"],
            forced_tool_ok=pf["forced_tool_ok"],preflight_error=pf["error"],
            served_max_model_len=spec.get("_served_max_model_len",CFG["max_model_len"]),
        )

        if not (pf["chat_ok"] and pf["auto_tool_ok"]):
            mark_model(slug,status="SKIPPED_TOOL_PREFLIGHT")
            print("SKIP: autonomous tool-call preflight failed.")
            return

        bf_st,bf_df,bf_rc=run_bfcl(spec)
        mark_model(slug,bfcl=bf_st,bfcl_returncode=bf_rc)
        print("BFCL:",bf_st,"rows:",len(bf_df))

        run_all=spec["full_bench"] or MODE=="FULL"
        if run_all:
            dj_st,dj_df,dj_ss=run_dojo(spec)
            mark_model(slug,agentdojo=dj_st)
            print("AgentDojo:",dj_st,"cases:",len(dj_df))

            tau_st,tau_df,tau_ss=run_tau(spec)
            mark_model(slug,tau3=tau_st)
            print("tau3:",tau_st,"trajectories:",len(tau_df))
        else:
            dj_st="SUPPLEMENTARY_NOT_RUN"
            tau_st="SUPPLEMENTARY_NOT_RUN"
            mark_model(slug,agentdojo=dj_st,tau3=tau_st)

        if run_all:
            complete=(
                str(bf_st).startswith("SUPPORTED") and
                str(dj_st).startswith("SUPPORTED") and
                str(tau_st).startswith("SUPPORTED")
            )
        else:
            complete=str(bf_st).startswith("SUPPORTED")

        mark_model(slug,status="COMPLETE" if complete else "PARTIAL_COMPLETE")

    except Exception as e:
        mark_model(slug,status="ERROR",error=repr(e))
        print("MODEL ERROR:",repr(e))
    finally:
        stop_server()
        if local_path is not None and not KEEP_MODEL_WEIGHTS:
            shutil.rmtree(local_path,ignore_errors=True)
        try:
            sync_recovery(f"{slug}_model_end")
        except Exception as e:
            print("Recovery sync warning:",repr(e))
        gc.collect()

    print("Checkpoint:",STATE.get(slug,{}))

print("✅ run_one_model() ready.")


In [ ]:
# CELL 7A — VERIFY PART 1 WAS RESTORED (FIXED V2)

STATE=load_checkpoint()

prior=["granite33_2b","qwen25_3b"]
missing=[x for x in prior if x not in STATE]

print("Restored checkpoint keys:",sorted(STATE.keys()))

if missing:
    raise RuntimeError(
        "Part 1 is genuinely unavailable after checking BOTH the rolling "
        "recovery ZIP and the complete Part-1 results ZIP. Missing keys: "
        + ", ".join(missing)
        + "\n\nExpected Drive files under:\n"
        "MyDrive/NTX_FINAL_3FAMILY_CLOSURE/\n"
        "  - NTX_FINAL_RECOVERY_LATEST.zip OR\n"
        "  - NTX_FINAL_3FAMILY_PART1_GRANITE_QWEN_RESULTS.zip\n\n"
        "If you downloaded Part 1 instead of keeping it in Drive, upload "
        "NTX_FINAL_3FAMILY_PART1_GRANITE_QWEN_RESULTS.zip to /content, "
        "then rerun Cell 1 and this cell."
    )

for slug in prior:
    s=STATE.get(slug,{})
    print(
        "\n",slug,
        "| status =",s.get("status"),
        "| BFCL =",s.get("bfcl"),
        "| AgentDojo =",s.get("agentdojo"),
        "| tau3 =",s.get("tau3"),
    )

print("\n✅ Part 1 state restored. Continue to Llama.")

In [ ]:
# CELL 7B — OPTIONAL PART 1 FILE CHECK

expected_patterns=[
    "bfcl_granite33_2b.csv",
    "bfcl_qwen25_3b.csv",
    "dojo_granite33_2b.csv",
    "dojo_qwen25_3b.csv",
    "tau_granite33_2b.csv",
    "tau_qwen25_3b.csv",
]

for name in expected_patterns:
    p=RESULTS/name
    print(name, "✅" if p.exists() else "—")

print(
    "\nMissing aggregate CSVs do not automatically mean failure; "
    "slice-level cached files and checkpoint status are also retained."
)


In [ ]:
# CELL 7C — PART 2 ONLY MODEL: Llama/Hermes AWQ
run_one_model("hermes3_llama31_8b_awq")

In [ ]:
# CELL 7D — ALL THREE CHECKPOINT STATUS

STATE=load_checkpoint()

for slug in [
    "granite33_2b",
    "qwen25_3b",
    "hermes3_llama31_8b_awq",
]:
    print("\n",slug)
    print(json.dumps(STATE.get(slug,{}),indent=2,default=str))


In [ ]:
# CELL 7E — GPU CLEANUP
stop_server()
gc.collect()
print(sh(["nvidia-smi"]).stdout)


In [ ]:
# CELL 7F — FORCE FINAL RECOVERY BACKUP
recovery=sync_recovery("PART2_AFTER_LLAMA")
print("Recovery bundle:",recovery)


In [ ]:
# CELL 7G — VERIFY ALL THREE MODEL KEYS EXIST

STATE=load_checkpoint()

required=[
    "granite33_2b",
    "qwen25_3b",
    "hermes3_llama31_8b_awq",
]

missing=[x for x in required if x not in STATE]

print("Present:",[x for x in required if x in STATE])
print("Missing:",missing)

if missing:
    print(
        "⚠️ Final gate will remain incomplete because one or more model "
        "checkpoints are absent."
    )
else:
    print("✅ All three model checkpoint keys are present.")


In [ ]:
# CELL 7H — CONTINUE TO FINAL AGGREGATION
print(
    "Proceed to Cell 8. "
    "The existing strict aggregation logic will determine whether "
    "the three-family closure claim is actually supported."
)


In [ ]:
# CELL 8 — RELOAD PERSISTED RESULTS + PAPER CLOSURE GATE

STATE=load_checkpoint()
status_rows=[]
bfcl_parts=[]
dojo_parts=[]
tau_parts=[]

for spec in MODEL_REGISTRY:
    s=STATE.get(spec["slug"],{})
    status_rows.append({
        "slug":spec["slug"],"family":spec["family"],"repo":spec["repo"],
        "counts_for_closure":spec["counts_for_closure"],
        "full_bench":spec["full_bench"],**s,
    })

    p=RESULTS/f"bfcl_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):bfcl_parts.append(d)
        except Exception:pass

    p=RESULTS/f"dojo_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):dojo_parts.append(d)
        except Exception:pass

    p=RESULTS/f"tau_{spec['slug']}.csv"
    if p.exists():
        try:
            d=pd.read_csv(p)
            if len(d):tau_parts.append(d)
        except Exception:pass

status_df=pd.DataFrame(status_rows)
bfcl_df=pd.concat(bfcl_parts,ignore_index=True) if bfcl_parts else pd.DataFrame()
dojo_df=pd.concat(dojo_parts,ignore_index=True) if dojo_parts else pd.DataFrame()
tau_df=pd.concat(tau_parts,ignore_index=True) if tau_parts else pd.DataFrame()

status_df.to_csv(RESULTS/"10_model_run_status.csv",index=False)
bfcl_df.to_csv(RESULTS/"20_bfcl_native_metrics.csv",index=False)
dojo_df.to_csv(RESULTS/"21_agentdojo_native_cases.csv",index=False)
tau_df.to_csv(RESULTS/"22_tau3_native_cases.csv",index=False)

display(status_df[[c for c in [
    "slug","family","status","bfcl","agentdojo","tau3","chat_ok","auto_tool_ok","error"
] if c in status_df.columns]])

summary_rows=[]
if len(bfcl_df):
    for (slug,family,metric),g in bfcl_df.groupby(["slug","family","metric"]):
        summary_rows.append({
            "benchmark":"BFCL-v4","slug":slug,"family":family,
            "metric":metric,"n":len(g),"mean":float(g.score.mean()),
        })
if len(dojo_df):
    for (slug,family),g in dojo_df.groupby(["slug","family"]):
        if g.utility.notna().any():
            summary_rows.append({
                "benchmark":"AgentDojo","slug":slug,"family":family,
                "metric":"utility","n":int(g.utility.notna().sum()),"mean":float(g.utility.mean()),
            })
        if g.security.notna().any():
            summary_rows.append({
                "benchmark":"AgentDojo","slug":slug,"family":family,
                "metric":"security","n":int(g.security.notna().sum()),"mean":float(g.security.mean()),
            })
if len(tau_df):
    for (slug,family),g in tau_df.groupby(["slug","family"]):
        summary_rows.append({
            "benchmark":"tau3","slug":slug,"family":family,
            "metric":"reward","n":int(g.reward.notna().sum()),"mean":float(g.reward.mean()),
        })

summary=pd.DataFrame(summary_rows)
summary.to_csv(RESULTS/"30_external_model_summary.csv",index=False)
display(summary)

complete_families=[]
closure_df=status_df[status_df["counts_for_closure"]==True]
for family,g in closure_df.groupby("family"):
    ok=False
    for _,r in g.iterrows():
        if (
            str(r.get("bfcl","")).startswith("SUPPORTED") and
            str(r.get("agentdojo","")).startswith("SUPPORTED") and
            str(r.get("tau3","")).startswith("SUPPORTED")
        ):
            ok=True
            break
    if ok:complete_families.append(family)

complete_families=sorted(set(complete_families))
closure_supported=len(complete_families)>=3

claims=pd.DataFrame([
    {"claim":"Local BFCL-v4 evidence","status":"SUPPORTED" if len(bfcl_df) else "MISSING"},
    {"claim":"Local AgentDojo evidence","status":"SUPPORTED" if len(dojo_df) else "MISSING"},
    {"claim":"Local tau3 evidence","status":"SUPPORTED" if len(tau_df) else "MISSING"},
    {
        "claim":"At least 3 independent T4 model families complete all required benchmarks",
        "status":"SUPPORTED" if closure_supported else "INCOMPLETE",
    },
    {
        "claim":"Split-run final three-family external paper-closure gate",
        "status":"SUPPORTED" if closure_supported else "INCOMPLETE",
    },
])
claims.to_csv(RESULTS/"31_claim_gate.csv",index=False)
display(claims)
print("Complete independent families:",complete_families)


In [ ]:
# CELL 9 — T4 PAPER TABLES / PROVENANCE — PAPER TABLES / FIGURES / PROVENANCE

summary.to_latex(
    PAPER/"local_external_summary.tex",
    index=False,float_format="%.4f"
)
claims.to_latex(
    PAPER/"local_external_claim_gate.tex",
    index=False
)
status_df.to_csv(PAPER/"local_model_status.csv",index=False)

if len(summary):
    for (benchmark,metric),g in summary.groupby(["benchmark","metric"]):
        gg=g.sort_values("mean")
        fig,ax=plt.subplots(figsize=(8,max(3,0.45*len(gg)+1)))
        ax.barh(gg["slug"],gg["mean"])
        ax.set_title(f"{benchmark}: {metric}")
        ax.set_xlabel(metric)
        fig.tight_layout()
        safe=re.sub(r"[^A-Za-z0-9]+","_",f"{benchmark}_{metric}")
        fig.savefig(PAPER/f"{safe}.png",dpi=220,bbox_inches="tight")
        plt.show()

if closure_supported:
    section=(
        "\\paragraph{External local-model validation.}\n"
        "We additionally evaluated the runtime using locally served open-weight models "
        "without relying on commercial inference APIs. At least three independent model "
        f"families completed BFCL-v4, AgentDojo, and $\\tau^3$: {', '.join(complete_families)}. "
        "Each model was served independently through a local OpenAI-compatible vLLM endpoint, "
        "and only native benchmark outputs with evaluated cases were retained. "
        "For $\\tau^3$, each run used the currently loaded model as both agent and user simulator; "
        "cross-family reward differences are therefore not interpreted as controlled head-to-head comparisons."
    )
else:
    section=(
        "\\paragraph{External local-model validation.}\n"
        "We evaluated a set of locally served open-weight models on BFCL-v4, AgentDojo, "
        "and $\\tau^3$. Because fewer than three independent model families completed all "
        "three benchmark families, these results are reported as bounded external evidence "
        "rather than a complete cross-family generalization claim. "
        "For $\\tau^3$, the same currently loaded model served as both agent and user simulator, "
        "so cross-family reward differences are not treated as controlled comparisons."
    )

(PAPER/"local_external_validation_section.tex").write_text(section+"\n")

# Model/revision provenance.
manifest={
    "experiment":"NTX-FINAL-3FAMILY-EXTERNAL-CLOSURE-SPLIT-RUN",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "mode":MODE,
    "gpu_name":GPU_NAME,
    "gpu_vram_gb":VRAM_GB,
    "benchmark_versions":BENCHMARK_VERSIONS,
    "tau_user_simulator":"same currently loaded local model as evaluated agent",
    "complete_families":complete_families,
    "closure_supported":closure_supported,
    "model_status":status_df.to_dict("records"),
    "config":CFG,
}
(RESULTS/"FINAL_MANIFEST.json").write_text(json.dumps(manifest,indent=2,default=str))

# Hash research outputs (not giant model weights).
hash_rows=[]
for label,folder in [("results",RESULTS),("paper",PAPER),("logs",LOGS)]:
    for p in sorted(folder.rglob("*")):
        if p.is_file() and p.name!="SHA256_MANIFEST.csv":
            h=hashlib.sha256()
            with open(p,"rb") as f:
                for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
            hash_rows.append({"file":f"{label}/{p.relative_to(folder)}","sha256":h.hexdigest()})
pd.DataFrame(hash_rows).to_csv(RESULTS/"SHA256_MANIFEST.csv",index=False)

In [ ]:
# CELL 10 — T4 FINAL ZIP / DOWNLOAD — ZIP ALL RESULTS AND DOWNLOAD

stage=BASE/"final_package"
if stage.exists():shutil.rmtree(stage)
stage.mkdir()

for src,name in [
    (RESULTS,"results"),
    (RAW,"raw_benchmark_outputs"),
    (LOGS,"logs"),
    (PAPER,"paper_integration"),
]:
    if src.exists():
        shutil.copytree(src,stage/name)

# Small reproducibility files only; do not package multi-GB model weights.
(stage/"README.txt").write_text(
    "NiyamTrace-X FINAL three-family external validation package.\n"
    "Model weights are intentionally excluded. See results/FINAL_MANIFEST.json "
    "and results/10_model_run_status.csv for model IDs/revisions/status.\n"
)

ZIP=ARCH/"NTX_FINAL_3FAMILY_EXTERNAL_CLOSURE_SPLIT_RESULTS.zip"
if ZIP.exists():ZIP.unlink()

with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in stage.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(stage)))

with zipfile.ZipFile(ZIP) as z:
    bad=z.testzip()

h=hashlib.sha256()
with open(ZIP,"rb") as f:
    for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)

print("Closure gate:", "SUPPORTED" if closure_supported else "INCOMPLETE")
print("Complete families:",complete_families)
print("ZIP integrity:", "PASS" if bad is None else bad)
print("ZIP:",ZIP)
print("SHA256:",h.hexdigest())

try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print("Auto-download unavailable:",repr(e))

if DRIVE_DIR is not None:
    target=DRIVE_DIR/ZIP.name
    shutil.copy2(ZIP,target)
    print("Final ZIP copied to Drive:",target)
